# 05 — Organize Clean Data
# Giai đoạn 1 — Mục 1.0 — Tổ chức 40 file sạch vào data/clean/

- **Đầu vào**: `outputs/tables/manifest_filtered.csv` (từ notebook 01)
- **Đầu ra**:
  - `data/clean/` — 40 file sạch
  - `outputs/tables/manifest_clean.csv` — manifest kèm cột `fs` (sampling rate thật)


In [1]:
from pathlib import Path
import shutil
import pandas as pd

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
CLEAN_DATA_ROOT = Path("../../data/clean")
CLEAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")


In [3]:
# Sampling rate thật của 4 file Normal (GĐ0 mục 0.1.4); các file lỗi đều 12 kHz
# TODO: nên chuyển mapping này vào common/config.py để tránh lặp lại giữa các notebook
NORMAL_FS_OVERRIDE = {
    "97_Normal_0.mat": 24000,
    "98_Normal_1.mat": 48000,
    "99_Normal_2.mat": 48000,
    "100_Normal_3.mat": 48000,
}

def detect_fs(file_path):
    return NORMAL_FS_OVERRIDE.get(Path(file_path).name, 12000)


In [4]:
rows = []
for _, row in manifest.iterrows():
    src = Path(row["file_path"])
    if not src.exists():
        print(f"Thiếu file: {src}")
        continue
    dst = CLEAN_DATA_ROOT / src.name
    shutil.copy2(src, dst)

    new_row = row.to_dict()
    new_row["file_path"] = str(dst)
    new_row["fs"] = detect_fs(src)
    rows.append(new_row)

manifest_clean = pd.DataFrame(rows)
manifest_clean.to_csv(TABLES_DIR / "manifest_clean.csv", index=False)
print(f"Đã tổ chức {len(manifest_clean)} file vào {CLEAN_DATA_ROOT}")
manifest_clean[["label", "load_hp", "fs"]].head()


Đã tổ chức 40 file vào ..\..\data\clean


,label,load_hp,fs
0,B,0,12000
1,B,1,12000
2,B,2,12000
3,B,3,12000
4,B,0,12000
